# Intro to Model and Data Drift

Once a model is deployed, the data it sees can slowly change. Sometimes the feature distribution shifts, sometimes the target behavior changes, and sometimes both happen together. This notebook shows how to compare two taxi datasets and turn that intuition into measurable drift signals.


```mermaid
flowchart LR
    A["Training month"] --> C["Compare distributions"]
    B["Current month"] --> C
    C --> D["Feature drift"]
    C --> E["Target drift"]
    C --> F["Prediction drift"]
    D --> G["Investigate or monitor"]
    E --> G
    F --> G
```


**Example:**

You've built a model that predicts how much a person will spend in an online webshop, using the age and the gender of a person. The training dataset mostly consists of women, with the average age of all customers being 35. The model learns that women and customers in their mid-thirties buy something more often. After monitoring the model's predictive performance, you notice that it's starting to decrease... But why? 

After doing some research you discovered that the webshop now has younger customers: the average age has dropped from 35 years to 25 years. Furthermore, the live data shows that men, not women, are now the majority!

So, the original relationship between the features that the model has learned from the original training dataset isn't valid anymore on the real-time incoming data, which causes the model to lose its predictive power.

**This loss in predictive power is called model drift and the change in the data is called data drift**.

Let's have a look at the NY Taxi dataset.

In [ ]:
import os
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from dotenv import load_dotenv

Let's start by loading the datasets. The first one is the dataset we train the model on:

In [ ]:
year = 2024
month = 1
color = "green"

df_2024_jan = pd.read_parquet(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")

The second one is the dataset we compare the first one to:

In [ ]:
year = 2025
month = 2
color = "green"

data_path = Path(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")
data_path.parent.mkdir(exist_ok=True)

if not data_path.exists():
    urlretrieve(
        f"https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month:02d}.parquet",
        data_path,
    )

df_2025_feb = pd.read_parquet(data_path)

In [ ]:
features = [
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "passenger_count",
    "fare_amount",
    "total_amount",
]
target = "duration"

In [ ]:
def calculate_trip_duration_in_minutes(df):
    df["duration"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]
    df = df[(df["passenger_count"] > 0) & (df["passenger_count"] < 8)]
    df = df[features + [target]]
    return df

In [ ]:
df_2024_jan = calculate_trip_duration_in_minutes(df_2024_jan)
df_2025_feb = calculate_trip_duration_in_minutes(df_2025_feb)

if len(df_2024_jan) < len(df_2025_feb):
    print("Trimming df_2025_feb to match length of df_2024_jan")
    df_2025_feb = df_2025_feb.sample(frac=1, random_state=42).reset_index(drop=True)
    df_2025_feb = df_2025_feb.iloc[: len(df_2024_jan)]
elif len(df_2024_jan) > len(df_2025_feb):
    print("Trimming df_2024_jan to match length of df_2025_feb")
    df_2024_jan = df_2024_jan.sample(frac=1, random_state=42).reset_index(drop=True)
    df_2024_jan = df_2024_jan.iloc[: len(df_2025_feb)]
else:
    print("Both DataFrames have the same length")

### Now let's compare the data

In [ ]:
df_2024_jan.describe()

In [ ]:
df_2025_feb.describe()

In [ ]:
columns = df_2024_jan.columns
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for ax, var in zip(axes.flat, columns):
    combined = pd.concat([df_2024_jan[var], df_2025_feb[var]])
    # Use the same bins for both months and skip the most extreme 1% of values,
    # so a few outliers don't squeeze the rest of the histogram.
    value_range = (combined.quantile(0.005), combined.quantile(0.995))
    ax.hist(
        df_2024_jan[var],
        bins=40,
        range=value_range,
        alpha=0.4,
        label="Jan 2024 (training)",
    )
    ax.hist(
        df_2025_feb[var],
        bins=40,
        range=value_range,
        histtype="step",
        linewidth=1.5,
        label="Feb 2025 (current)",
    )
    ax.set_title(var)

# Seven columns fill seven of the eight panels, so hide the empty one.
for ax in axes.flat[len(columns) :]:
    ax.set_visible(False)

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

We see small changes in nearly all features. But we can also calculate the difference between the two datasets with:

- **Jensen-Shannon distance** for categorical features,
- **Wasserstein distance** for numerical features.


For categorical features, we first convert location IDs into normalized frequency distributions, then compare those distributions. The lower the value, the more similar the two datasets are.

Let's calculate the distance between the features of the two datasets:


In [ ]:
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance


def drift_jensenshannon_test(var_train: pd.Series, var_test: pd.Series) -> float:
    """Measure distance between categorical probability distributions."""
    categories = sorted(set(var_train.dropna()) | set(var_test.dropna()))
    train_distribution = var_train.value_counts(normalize=True).reindex(
        categories, fill_value=0
    )
    test_distribution = var_test.value_counts(normalize=True).reindex(
        categories, fill_value=0
    )
    return float(jensenshannon(train_distribution, test_distribution))


def drift_wasserstein_test(var_train: pd.Series, var_test: pd.Series) -> float:
    """Measure distance between two numerical distributions."""
    return float(wasserstein_distance(var_train, var_test))

In [ ]:
df_drift = pd.DataFrame(
    {
        "feature": pd.Series(dtype="object"),
        "distance": pd.Series(dtype="float64"),
        "type": pd.Series(dtype="object"),
    }
)

cat_features = ["PULocationID", "DOLocationID"]
num_features = ["trip_distance", "passenger_count", "fare_amount", "total_amount"]

for feature in num_features:
    df_drift = pd.concat(
        [
            df_drift,
            pd.DataFrame(
                [
                    {
                        "feature": feature,
                        "distance": drift_wasserstein_test(
                            df_2024_jan[feature], df_2025_feb[feature]
                        ),
                        "type": "Wasserstein",
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

for feature in cat_features:
    df_drift = pd.concat(
        [
            df_drift,
            pd.DataFrame(
                [
                    {
                        "feature": feature,
                        "distance": drift_jensenshannon_test(
                            df_2024_jan[feature], df_2025_feb[feature]
                        ),
                        "type": "Jensen-Shannon",
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

There are more **tests to detect drift**, but this is a good start. If you want to learn more about **drift detection**, check out this [article](https://www.evidentlyai.com/blog/data-drift-detection-large-datasets). In practice, thresholds should be calibrated for the data and business context. For this lesson, we use `0.1` as a simple rule of thumb so we can practice turning distances into a drift flag:

In [ ]:
df_drift["True/False Drift"] = df_drift["distance"] > 0.1
df_drift

It seems we can confirm what we saw in the histograms: there is drift in half of our features. Using this method **we can also detect drift in the target variable**. Let's have a look at the target variable:

In [ ]:
drift_wasserstein_test(df_2024_jan[target], df_2025_feb[target])

So we also have a drift in our target variable.

We can also check whether the model predictions themselves are drifting. For that step we load the registered model created in notebook 01 and score both datasets with the same feature set.


In [ ]:
# Load environment variables from .env file
load_dotenv()

# Get the MLflow tracking URI from environment variables
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")

# Set up the connection to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("green-taxi-monitoring")

In [ ]:
def load_model(model_name):
    alias = "production"
    model_uri = f"models:/{model_name}@{alias}"
    model = mlflow.pyfunc.load_model(model_uri)
    return model

In [ ]:
model_name = "green-taxi-ride-duration"

loaded_model = load_model(model_name)

In [ ]:
df_2024_jan["prediction"] = loaded_model.predict(df_2024_jan[features])

In [ ]:
df_2025_feb["prediction"] = loaded_model.predict(df_2025_feb[features])

Now we can calculate the drift in the predictions:

In [ ]:
drift_wasserstein_test(df_2024_jan["prediction"], df_2025_feb["prediction"])

And also here we can see drift. That does not automatically mean the model is unusable, but it is a strong signal that the training distribution and the current production-like batch are no longer aligned. In practice, this is the kind of evidence that can trigger retraining or a closer investigation.


In the next notebook we will use Evidently to automate these checks and package them into a reusable report.
